# PS-S3 — Trotter error versus processor noise

Phase 1 submission for Qiskit Fall Fest SRMAP 2026. This notebook reads the saved benchmark and can regenerate it.

**Selected problem:** S3, Noise in Trotter Simulation.
**Claim:** Processor B hosts the periodic 4-spin interaction with zero SWAPs. Processor A is a tree, so the same circuit pays a routing tax of exactly three CX gates per SWAP. Under one shared noise model, that tax lowers fidelity. Useful depth peaks at 4 Trotter steps.

No quantum advantage is claimed. The reference is exact diagonalization of a 16-dimensional Hamiltonian.


## Reproduce

Run `python main.py` from the repository root to overwrite `results/`. The cells below only read the recorded run.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results" / "tables" / "all_runs.csv").exists():
    ROOT = Path(".")
df = pd.read_csv(ROOT / "results" / "tables" / "all_runs.csv")
primary = df[(df.order == 1) & (df.time == 1.0) & (df.seed == 42)].drop_duplicates(["processor", "steps"])
cols = ["processor", "steps", "exact_z0", "ideal_trotter_z0", "noisy_z0", "noisy_abs_error", "cx", "swap_count", "depth", "noisy_logical_fidelity"]
primary[cols].sort_values(["processor", "steps"]).round(4)


## Figures

The fidelity curve is the result to remember. Ideal fidelity keeps rising. Noisy fidelity peaks at M = 4, and Processor B stays above Processor A once the formula is no longer the dominant error.


In [ ]:
from IPython.display import Image, display, Markdown
fig = Path("results/figures")
for name, note in [
    ("processors.png", "Official coupling graphs. A has no cycle. B contains the 4-cycle 1-2-5-6."),
    ("fidelity_vs_steps.png", "State fidelity with the exact evolved state."),
    ("error_vs_steps.png", "Absolute error in the requested observable."),
    ("resources_vs_steps.png", "CX and SWAP counts. A's extra CX equals 3 times its SWAP count."),
    ("observable_vs_time.png", "Fixed M = 4 across the requested time window."),
    ("error_vs_cx.png", "Same observable error against executed CX count."),
]:
    display(Markdown(f"**{note}**"))
    display(Image(filename=str(fig / name)))


## Interpretation

At M = 1 both processors fail for the same reason: the product formula is inaccurate. From M = 2 upward, architecture separates them. Eight steps improves the ideal state and worsens the noisy state. A later processor should contain the Hamiltonian's interaction graph as a subgraph. Adding degree without hosting the periodic bond would not be supported by this measurement.

Limitations and the noise-model assumptions are in `report.md`. AI assistance is disclosed in `AI_DISCLOSURE.md`.
